## Install and import libraries

In [1]:
# Install imbalanced-learn which contains SMOTE
import subprocess
subprocess.run(["pip", "install", "imbalanced-learn"])

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import pickle
import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]


## Load the Combined Dataset

In [5]:
# Load the combined dataset we created in notebook 06
df = pd.read_csv('data/combined_reviews.csv')

print(f"Total reviews: {len(df)}")
print(f"\nSentiment distribution before SMOTE:")
print(df['sentiment'].value_counts())

Total reviews: 6287

Sentiment distribution before SMOTE:
sentiment
negative    3122
positive    2065
neutral     1100
Name: count, dtype: int64


## Prepare and apply TF-IDF

In [7]:
# X is our input text
# y is what we want to predict
X = df['clean_text']
y = df['sentiment']

# Split into train and test 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y    # keep sentiment balance in both sets
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Convert text to numbers using TF-IDF
# We fit ONLY on training data — never on test data
# This prevents data leakage
tfidf = TfidfVectorizer(
    max_features=15000,   # increased from 10000 for more vocabulary coverage
    ngram_range=(1, 2),   # single words and pairs of words
    sublinear_tf=True     # reduces weight of very common words
)

# fit_transform learns the vocabulary and converts training text to numbers
X_train_tfidf = tfidf.fit_transform(X_train)

# transform only converts test text using vocabulary learned from training
X_test_tfidf = tfidf.transform(X_test)

print(f"\nTF-IDF matrix shape: {X_train_tfidf.shape}")

Training samples: 5029
Testing samples: 1258

TF-IDF matrix shape: (5029, 15000)


## Apply SMOTE to fix class imbalance

In [9]:
# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)

# fit_resample generates synthetic samples for underrepresented classes
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_tfidf,
    y_train
)

print("Sentiment distribution AFTER SMOTE:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for sentiment, count in zip(unique, counts):
    print(f"{sentiment}: {count}")

print(f"\nTraining samples before SMOTE: {len(X_train)}")
print(f"Training samples after SMOTE:  {X_train_balanced.shape[0]}")

Sentiment distribution AFTER SMOTE:
negative: 2497
neutral: 2497
positive: 2497

Training samples before SMOTE: 5029
Training samples after SMOTE:  7491


## Train improved TF-IDF model

In [10]:
# Train Logistic Regression on the balanced dataset
# Notice we no longer need class_weight='balanced'
# because SMOTE already balanced the classes for us
improved_tfidf_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

improved_tfidf_model.fit(X_train_balanced, y_train_balanced)

# Evaluate on test set
y_pred = improved_tfidf_model.predict(X_test_tfidf)

print("=" * 50)
print("IMPROVED TF-IDF + LOGISTIC REGRESSION")
print("=" * 50)
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")

IMPROVED TF-IDF + LOGISTIC REGRESSION
              precision    recall  f1-score   support

    negative       0.89      0.93      0.91       625
     neutral       0.70      0.66      0.68       220
    positive       0.94      0.91      0.92       413

    accuracy                           0.87      1258
   macro avg       0.84      0.83      0.84      1258
weighted avg       0.87      0.87      0.87      1258

Accuracy: 87.4%


## Build nlptown BERT model

In [11]:
from transformers import pipeline

# Load the nlptown model
nlptown_classifier = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512
)

print("Model loaded successfully!")

# Test it on a quick example
test_review = "The flight was delayed and the staff were very rude"
result = nlptown_classifier(test_review)
print(f"\nTest prediction: {result}")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded successfully!

Test prediction: [{'label': '1 star', 'score': 0.4951740503311157}]


## Evaluate nlptown BERT

In [12]:
# nlptown predicts 1-5 stars so we need to convert to our 3 class system
def nlptown_to_sentiment(label):
    # label comes back as '1 star', '2 stars' etc
    stars = int(label.split()[0])
    if stars <= 2:
        return 'negative'
    elif stars == 3:
        return 'neutral'
    else:
        return 'positive'

print("Running nlptown BERT on test set...")
print("This will take a few minutes...")

# Run predictions on test set in batches for speed
# We use only first 500 test samples to save time
test_sample = X_test[:500].tolist()
predictions = nlptown_classifier(test_sample, batch_size=16)

# Convert predictions to our sentiment labels
y_pred_nlptown = [
    nlptown_to_sentiment(pred['label'])
    for pred in predictions
]

y_test_sample = y_test[:500].tolist()

print("=" * 50)
print("NLPTOWN BERT RESULTS")
print("=" * 50)
print(classification_report(y_test_sample, y_pred_nlptown))
print(f"Accuracy: {accuracy_score(y_test_sample, y_pred_nlptown)*100:.1f}%")

Running nlptown BERT on test set...
This will take a few minutes...
NLPTOWN BERT RESULTS
              precision    recall  f1-score   support

    negative       0.73      0.97      0.83       244
     neutral       0.29      0.25      0.27        83
    positive       0.96      0.57      0.71       173

    accuracy                           0.71       500
   macro avg       0.66      0.60      0.60       500
weighted avg       0.73      0.71      0.70       500

Accuracy: 71.0%


## Final Model Comparison

In [13]:
print("=" * 50)
print("FINAL MODEL COMPARISON")
print("=" * 50)
print(f"VADER Baseline:                    66.0%")
print(f"Original TF-IDF + LR:              74.8%")
print(f"Original DistilBERT:               78.5%")
print(f"Improved TF-IDF + LR + SMOTE:      {accuracy_score(y_test, y_pred)*100:.1f}%")
print(f"nlptown BERT:                      {accuracy_score(y_test_sample, y_pred_nlptown)*100:.1f}%")

FINAL MODEL COMPARISON
VADER Baseline:                    66.0%
Original TF-IDF + LR:              74.8%
Original DistilBERT:               78.5%
Improved TF-IDF + LR + SMOTE:      87.4%
nlptown BERT:                      71.0%


## Save the improved model

In [14]:
import pickle

os.makedirs('../models', exist_ok=True)

# Save the improved TF-IDF model as a pipeline
# We need to save both the vectorizer and the model together
improved_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('clf',   improved_tfidf_model)
])

# We need to retrain the pipeline version on balanced data
# So we rebuild it cleanly
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train on original unbalanced data using class_weight for the pipeline version
final_pipeline.fit(X_train, y_train)

# Save the pipeline
with open('../models/improved_sentiment_model.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)

print("Improved TF-IDF model saved to models/improved_sentiment_model.pkl")
print("nlptown BERT loads directly from HuggingFace — no saving needed!")

Improved TF-IDF model saved to models/improved_sentiment_model.pkl
nlptown BERT loads directly from HuggingFace — no saving needed!
